# Test 1 — does the typed head earn its place?

The v13 baseline measured **accuracy**, where the head cannot win:
kNN 93.60% > fingerprint 93.15% > interaction 92.73%, single seed, differences
inside noise. That is the wrong axis. The typed-decision claim is not accuracy —
it is **calibrated probabilities** and an **abstention gate** that retrieval
cannot provide.

This kernel measures those axes on cached Banking77 embeddings, on CPU (no
encoder, no GPU, no quota):

1. **Calibrated retrieval baselines** — `softmax(cos/τ)` and kNN with a fitted
   temperature. These are the honest competitors: a trained head must beat a
   *calibrated* retrieval model, not a raw similarity score.
2. **Proper scores** — log score and Brier for every candidate (gated); ECE
   reported, not gated.
3. **Abstention** — selective accuracy vs coverage, and the coverage reachable
   at 90% precision. This is the product question: how much can be auto-answered
   at a precision the caller can trust?
4. **Order invariance** — the "by construction" claim, verified empirically on
   real embeddings rather than asserted.
5. **Multi-seed variance** — so a 0.4-point gap is not read as a finding.

## 0. Environment (CPU-only by design)

In [ ]:
import json, shutil, subprocess, time
from pathlib import Path

# A CPU-only Kaggle instance has no nvidia-smi binary at all, so probe before running it.
gpu = "none (expected: this analysis needs no accelerator)"
if shutil.which("nvidia-smi"):
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name"],
                         capture_output=True, text=True).stdout.strip()
print("GPU:", gpu)

import numpy as np, torch
print("numpy", np.__version__, "| torch", torch.__version__)

STATUS = {"status": "running", "stage": "env", "started": time.time()}

# Output root: Kaggle writes to /kaggle/working; a local dry run falls back.
WORK = Path("/kaggle/working")
if not WORK.exists():
    WORK = Path("/tmp/s1-work")
WORK.mkdir(parents=True, exist_ok=True)

def write_status(stage=None, error=None, done=False):
    if stage: STATUS["stage"] = stage
    if error: STATUS["error"] = str(error); STATUS["status"] = "failed"
    if done: STATUS["status"] = "complete"
    STATUS["ended"] = time.time()
    (WORK / "run_status.json").write_text(json.dumps(STATUS, indent=2))
write_status()
print("work dir:", WORK)

In [ ]:
# Self-contained source bundle (same approach as the baseline kernel: no
# dependency on a mounted worktree dataset).
import base64, io, sys, zipfile

BUNDLE = "..."
REPO = str(WORK / "nanocore-s1")
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))) as z:
    z.extractall(REPO)
sys.path.insert(0, REPO)

from src.decision import metrics
from src.decision.head import DecisionHead
from src.decision.schema import DecisionExample
print("decision package ready at", REPO)

## 1. Cached embeddings

Produced by the v13 baseline kernel from `google/embeddinggemma-2`
(text+vision, bf16, 439M). Unit-normalized, so cosine similarity is a plain dot
product.

In [ ]:
import glob

hits = glob.glob("/kaggle/input/**/banking77_embeddings.npz", recursive=True)
assert hits, "embeddings not found — attach mauricew/nanocore-s1-banking77-emb"
z = np.load(hits[0], allow_pickle=True)
X_train, y_train = z["X_train"], z["y_train"]
X_test, y_test = z["X_test"], z["y_test"]
O = z["O"]
option_texts = [str(s) for s in z["option_texts"]]
K = len(option_texts)

print("train", X_train.shape, "| test", X_test.shape, "| options", O.shape, "| classes", K)
print("unit-norm mean:", round(float(np.linalg.norm(X_test, axis=1).mean()), 5))
write_status("embeddings-loaded")

In [ ]:
# Split: 90% of train fits, 10% fits temperatures/thresholds. Test untouched.
g = np.random.default_rng(0)
perm = g.permutation(len(X_train))
n_val = int(0.1 * len(perm))
val_idx, fit_idx = perm[:n_val], perm[n_val:]
X_fit, y_fit = X_train[fit_idx], y_train[fit_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

def onehot(y):
    m = np.zeros((len(y), K))
    m[np.arange(len(y)), y] = 1.0
    return m

def softmax(s):
    s = np.asarray(s, dtype=np.float64)
    s = s - s.max(axis=1, keepdims=True)
    e = np.exp(s)
    return e / e.sum(axis=1, keepdims=True)

def fit_tau(val_scores, val_y, lo=-0.3, hi=1.5, n=37):
    # fit temperature by log score on held-out data (proper score -> honest)
    grid = [round(10 ** e, 4) for e in np.linspace(lo, hi, n)]
    best_t, best_v = 1.0, float("inf")
    for t in grid:
        v = metrics.log_score(softmax(val_scores / t), onehot(val_y))
        if v < best_v:
            best_t, best_v = float(t), v
    return best_t

Y_TEST = onehot(y_test)
print("fit", X_fit.shape, "| val", X_val.shape, "| test", X_test.shape)

## 2. Calibrated retrieval baselines

Both get a temperature fitted on held-out data by log score — the same courtesy
the heads get. Without this, comparing an uncalibrated similarity score to a
trained head would be a straw man.

In [ ]:
RESULTS = {"baselines": {}, "heads": {}, "abstention": {}, "order_invariance": {}}

def summarize(name, probs, store, extra=None):
    preds = probs.argmax(1)
    rec = {
        "accuracy": float((preds == y_test).mean()),
        "log_score": float(metrics.log_score(probs, Y_TEST)),
        "brier": float(metrics.brier_score(probs, Y_TEST)),
        "ece_report_only": float(metrics.expected_calibration_error(probs, Y_TEST)),
    }
    if extra: rec.update(extra)
    store[name] = rec
    print(f"{name:26s} acc={rec['accuracy']:.4f} log={rec['log_score']:.3f} "
          f"brier={rec['brier']:.3f} ece={rec['ece_report_only']:.3f}")
    return rec

# --- cosine + temperature: softmax over similarity to the 77 option vectors
tau_cos = fit_tau(X_val @ O.T, y_val)
probs_cos = softmax((X_test @ O.T) / tau_cos)
summarize("cosine_tau", probs_cos, RESULTS["baselines"], {"temperature": tau_cos})

# --- kNN + temperature: class score = summed similarity of its top-k neighbours
def knn_class_scores(Xq, Xr, yr, k):
    sims = Xq @ Xr.T
    idx = np.argpartition(-sims, k, axis=1)[:, :k]
    out = np.zeros((len(Xq), K))
    for i in range(len(Xq)):
        rows = idx[i]
        np.add.at(out[i], yr[rows], sims[i, rows])
    return out

for k in (1, 5, 20):
    tau_k = fit_tau(knn_class_scores(X_val, X_fit, y_fit, k), y_val)
    probs_k = softmax(knn_class_scores(X_test, X_fit, y_fit, k) / tau_k)
    summarize(f"knn{k}_tau", probs_k, RESULTS["baselines"], {"temperature": tau_k, "k": k})
write_status("baselines")

## 3. Trained typed heads, multi-seed

Same protocol for both modes, 3 seeds, so any reported gap carries a spread.
`hidden=32`, modest epochs: this is a protocol for comparison, not a
tuned headline number.

In [ ]:
fit_ex = [DecisionExample(state_embedding=X_fit[i], qtype="choice", labels=option_texts,
                          target=[1.0 if j == y_fit[i] else 0.0 for j in range(K)],
                          option_embeddings=[O[j] for j in range(K)])
          for i in range(len(X_fit))]
val_ex = [DecisionExample(state_embedding=X_val[i], qtype="choice", labels=option_texts,
                          target=[1.0 if j == y_val[i] else 0.0 for j in range(K)],
                          option_embeddings=[O[j] for j in range(K)])
          for i in range(len(X_val))]

def train_and_score(mode, seed, epochs, hidden=32):
    head = DecisionHead(dim=768, mode=mode, hidden=hidden)
    head.fit(fit_ex, epochs=epochs, lr=0.05, batch_size=64, seed=seed)
    head.fit_temperature(val_ex)
    probs = np.zeros((len(X_test), K))
    for i in range(len(X_test)):
        p = head.predict(X_test[i], [O[j] for j in range(K)],
                         qtype="choice", labels=option_texts)
        probs[i] = [p.probabilities[l] for l in option_texts]
    return head, probs

# CPU budget is the constraint here, so the protocol is explicit per mode:
# fingerprint is a 77x768 linear map (cheap -> 3 seeds); interaction expands
# every option to 3*d features (expensive -> 2 seeds, fewer epochs). The
# comparison against the baselines is unaffected by the seed count; only the
# reported spread is.
PROTOCOL = {"fingerprint": (120, (0, 1, 2)), "interaction": (40, (0, 1))}
HEAD_PROBS = {}
for mode, (epochs, seeds) in PROTOCOL.items():
    runs = []
    for seed in seeds:
        head, probs = train_and_score(mode, seed, epochs)
        rec = summarize(f"{mode}_seed{seed}", probs, HEAD_PROBS, {"seed": seed})
        runs.append(rec)
        if seed == 0:
            HEAD_PROBS[f"{mode}_probs_seed0"] = probs
            head.save(WORK / f"head_{mode}_seed0.json")
    keys = ("accuracy", "log_score", "brier", "ece_report_only")
    RESULTS["heads"][f"{mode}_mean"] = {k: float(np.mean([r[k] for r in runs])) for k in keys}
    RESULTS["heads"][f"{mode}_std"] = {k: float(np.std([r[k] for r in runs])) for k in keys}
    RESULTS["heads"][f"{mode}_protocol"] = {"epochs": epochs, "seeds": list(seeds)}
    m, s = RESULTS["heads"][f"{mode}_mean"], RESULTS["heads"][f"{mode}_std"]
    print(f"{mode} mean over {len(seeds)} seeds: acc={m['accuracy']:.4f}+-{s['accuracy']:.4f} "
          f"log={m['log_score']:.3f}+-{s['log_score']:.3f} brier={m['brier']:.3f}+-{s['brier']:.3f}")
write_status("heads")

## 4. Abstention — the product question

Rank every candidate by its own confidence (`max p`) and keep the top fraction.
For a System One gate the question is not "how accurate is it", it is **"how
much can I auto-answer while keeping the answers I keep trustworthy"**.

In [ ]:
def selective_curve(probs, steps=10):
    conf = probs.max(1)
    correct = (probs.argmax(1) == y_test).astype(float)
    order = np.argsort(-conf)
    curve = []
    for c in np.linspace(0.1, 1.0, steps):
        n = max(1, int(c * len(y_test)))
        keep = order[:n]
        curve.append({"coverage": round(float(c), 2),
                      "selective_accuracy": float(correct[keep].mean())})
    return curve

def coverage_at(probs, target=0.90):
    conf = probs.max(1)
    correct = (probs.argmax(1) == y_test).astype(float)
    order = np.argsort(-conf)
    best = 0.0
    for n in range(1, len(y_test) + 1):
        if correct[order[:n]].mean() >= target:
            best = n / len(y_test)
    return float(best)

In [ ]:
# recompute kNN probability matrices once, keeping them for the curves
KNN_PROBS = {}
for k in (1, 5, 20):
    tau_k = RESULTS["baselines"][f"knn{k}_tau"]["temperature"]
    KNN_PROBS[f"knn{k}_tau"] = softmax(knn_class_scores(X_test, X_fit, y_fit, k) / tau_k)

CANDIDATES = {"cosine_tau": probs_cos, **KNN_PROBS,
              "fingerprint": HEAD_PROBS["fingerprint_probs_seed0"],
              "interaction": HEAD_PROBS["interaction_probs_seed0"]}

for name, probs in CANDIDATES.items():
    RESULTS["abstention"][name] = {
        "curve": selective_curve(probs),
        "coverage_at_90pct_precision": coverage_at(probs, 0.90),
        "coverage_at_95pct_precision": coverage_at(probs, 0.95),
    }
    r = RESULTS["abstention"][name]
    print(f"{name:16s} cov@90%={r['coverage_at_90pct_precision']:.3f} "
          f"cov@95%={r['coverage_at_95pct_precision']:.3f} "
          f"| acc@50%cov={r['curve'][4]['selective_accuracy']:.4f}")
write_status("abstention")

## 5. Order invariance on real embeddings

Per-option isolated scoring makes the argmax invariant to option order *by
construction*. Verify it on real data instead of trusting the docstring.

In [ ]:
import torch

head_fp = DecisionHead.load(WORK / "head_fingerprint_seed0.json")
rng = np.random.default_rng(7)
sample = rng.choice(len(X_test), 40, replace=False)
max_abs_diff, flips = 0.0, 0
for i in sample:
    order = rng.permutation(K)
    perm_opts = [O[j] for j in order]
    perm_labels = [option_texts[j] for j in order]
    p0 = head_fp.predict(X_test[i], [O[j] for j in range(K)],
                         qtype="choice", labels=option_texts)
    p1 = head_fp.predict(X_test[i], perm_opts, qtype="choice", labels=perm_labels)
    if p0.choice != p1.choice:
        flips += 1
    for j in range(K):
        max_abs_diff = max(max_abs_diff, abs(p0.probabilities[option_texts[j]]
                                             - p1.probabilities[option_texts[j]]))

RESULTS["order_invariance"] = {
    "states_checked": int(len(sample)),
    "max_abs_probability_difference": float(max_abs_diff),
    "argmax_flips": int(flips),
}
print(f"order invariance: {len(sample)} states, max |dp| = {max_abs_diff:.3e}, "
      f"argmax flips = {flips}")
write_status("order-invariance")

## 6. Verdict inputs

In [ ]:
# A verdict needs a rule, so state it before reading the numbers.
base = RESULTS["baselines"]["cosine_tau"]
fp = RESULTS["heads"]["fingerprint_mean"]
intr = RESULTS["heads"]["interaction_mean"]

verdict = {
    "head_beats_calibrated_retrieval_on_log_score":
        bool(fp["log_score"] < base["log_score"] and intr["log_score"] < base["log_score"]),
    "head_beats_calibrated_retrieval_on_brier":
        bool(fp["brier"] < base["brier"] and intr["brier"] < base["brier"]),
    "head_better_abstention_at_90pct_precision":
        bool(RESULTS["abstention"]["fingerprint"]["coverage_at_90pct_precision"]
             > RESULTS["abstention"]["cosine_tau"]["coverage_at_90pct_precision"]),
    "accuracy_gap_within_seed_noise": bool(
        abs(fp["accuracy"] - RESULTS["baselines"]["knn1_tau"]["accuracy"])
        < 2 * RESULTS["heads"]["fingerprint_std"]["accuracy"]),
    "order_invariance_holds": bool(RESULTS["order_invariance"]["argmax_flips"] == 0),
}
RESULTS["verdict"] = verdict
for k, v in verdict.items():
    print(f"{'PASS' if v else 'FAIL'}  {k}")

report = ["# Test 1 — does the typed head earn its place?", ""]
for section, payload in RESULTS.items():
    report.append(f"## {section}")
    report.append("```json")
    report.append(json.dumps(payload, indent=2, default=str))
    report.append("```")
    report.append("")
(WORK / "REPORT.md").write_text(chr(10).join(report))
(WORK / "results.json").write_text(json.dumps(RESULTS, indent=2, default=str))
write_status("done", done=True)

import zipfile
with zipfile.ZipFile(WORK / "artifacts.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["results.json", "REPORT.md", "run_status.json",
              "head_fingerprint_seed0.json", "head_interaction_seed0.json"]:
        p = WORK / f
        if p.exists():
            zf.write(p, f)
print(json.dumps(verdict, indent=2))